In [1]:
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import os
import glob
import torchvision.models as models

In [2]:
# Set the device to use (e.g., 'cpu', 'cuda', 'mps')
device = torch.device("mps" if torch.backends.mps.is_available() else
                      ("cuda" if torch.cuda.is_available() else "cpu"))

# Select Device According to Availability
print("Device selected:", device)

# If the device is CUDA, print the device capability
if device.type == "cuda":
    os.system("nvidia-smi")
    print()
    print("Device type:", device.type)
    print("Capability:", torch.cuda.get_device_capability(device))
else:
    print("Device capabilities are limited on MPSs and CPUs.")
    
    
# device = torch.device("cpu")

Device selected: cuda
Sun May 26 12:40:01 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.129.03             Driver Version: 535.129.03   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla P100-PCIE-16GB           Off | 00000000:00:04.0 Off |                    0 |
| N/A   34C    P0              26W / 250W |      2MiB / 16384MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+---

In [3]:
!wget https://download.pytorch.org/models/resnet50-19c8e357.pth /kaggle/working/resnet.pth

--2024-05-26 12:40:02--  https://download.pytorch.org/models/resnet50-19c8e357.pth
Resolving download.pytorch.org (download.pytorch.org)... 99.86.38.72, 99.86.38.37, 99.86.38.106, ...
Connecting to download.pytorch.org (download.pytorch.org)|99.86.38.72|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 102502400 (98M) [application/octet-stream]
Saving to: 'resnet50-19c8e357.pth'

resnet50-19c8e357.p 100%[===================>]  97.75M   267MB/s    in 0.4s    

2024-05-26 12:40:02 (267 MB/s) - 'resnet50-19c8e357.pth' saved [102502400/102502400]

/kaggle/working/resnet.pth: Scheme missing.
FINISHED --2024-05-26 12:40:02--
Total wall clock time: 0.4s
Downloaded: 1 files, 98M in 0.4s (267 MB/s)


In [4]:
class ResNetAnomaly(nn.Module):
    def __init__(self):
        super(ResNetAnomaly, self).__init__()

        # Load the pre-trained ResNet50 model without fine-tuning
        self.resnet = models.resnet50(weights=None)
        self.resnet.load_state_dict(torch.load('/kaggle/working/resnet50-19c8e357.pth'))

        # Get the number of features from the last layer of the ResNet
        num_ftrs = self.resnet.fc.in_features

        # Modify the last layer to output 2048 features
        self.resnet.fc = nn.Linear(num_ftrs, 2048)

        # Define additional layers for feature learning
        self.additional_layers = nn.Sequential(
            nn.Linear(2048, 1024),
            nn.ReLU(inplace=True),
        )

        # Attention mechanism
        self.attention = nn.Sequential(
            nn.Linear(2048, 2048),
            nn.Tanh(),
            nn.Linear(2048, 1),
            nn.Softmax(dim=1)
        )

    def forward(self, x):
        x = x.to(device)
        x = self.resnet(x)
        
        # Attention mechanism
        attention_weights = self.attention(x)
        attended_features = torch.matmul(attention_weights.unsqueeze(dim=2), x.unsqueeze(dim=1)).squeeze(dim=1)
        
        # Pass the attended features through the additional layers for further learning
        x = self.additional_layers(attended_features)

        return x
    
modelResnet = ResNetAnomaly()
modelResnet.to(device)
modelResnet.eval()
modelResnet

ResNetAnomaly(
  (resnet): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (downsample): Sequential(
     

In [6]:
def get_frames(path):
    video = cv2.VideoCapture(path)
    if not video.isOpened():
        raise IOError("Error reading video file")

    frame_count = int(video.get(cv2.CAP_PROP_FRAME_COUNT))

    frames = []
    for i in range(0, frame_count, 15):
        ret, frame = video.read()
        if ret:
            frame = cv2.resize(frame, (224, 224))
            frame = frame.astype("float32") / 255.0
            frames.append(frame)
    video.release()
    return np.array(frames)

In [7]:
video_path = "/kaggle/input/newtestdata/testData1/*/*.mp4"

video_path = glob.glob(video_path)

for video in video_path:
    frames_array = get_frames(video)
    print(frames_array.shape)
    frames_tensor = torch.tensor(frames_array).permute(0, 3, 1, 2).to(device)
    del frames_array
    
    # Get the Label
    labelName = video.split("/")[5]
    label = setLabel(labelName)
    print(labelName, label)
    # convert into numerical label
    
    
    # Embed the video
    videoEmbedding = modelResnet(frames_tensor)
    del frames_tensor
    print(videoEmbedding.shape)
    
    # After everything===
    
    del videoEmbedding
    
    
    
    
    
    
#     # (frames, 1024) e.g. (1124, 1024)
    
#     segments = embeddings.shape[0] // 100
    
    
#     for segment in segments:
#         # Get the segment
#         segment = embeddings[segment*100:(segment+1)*100]
#         # Segment shape -> (100, 1024)
        
#         # Model will take input shape (8, 100, 1024)
#         segment = segment.unsqueeze(0)
        
#         # make 8 batch is necessary
        
#         # Predict the segment label
#         outputs = sru(segment)
        
#         _, predicted = torch.max(outputs.data, 1)
#         # total += labels.size(0)
#         # correct += (predicted == labels).sum().item()
#         # match the predicted label with the original label
        
#         # Print predicted label for each segment
        
#     print(f"Current video label: {label}")
        



(172, 224, 224, 3)
roadaccidents 4
torch.Size([172, 1024])
(151, 224, 224, 3)
roadaccidents 4
torch.Size([151, 1024])
(162, 224, 224, 3)
roadaccidents 4
torch.Size([162, 1024])
(160, 224, 224, 3)
roadaccidents 4
torch.Size([160, 1024])
(160, 224, 224, 3)
roadaccidents 4
torch.Size([160, 1024])
(176, 224, 224, 3)
roadaccidents 4
torch.Size([176, 1024])
(160, 224, 224, 3)
roadaccidents 4
torch.Size([160, 1024])
(169, 224, 224, 3)
assault 3
torch.Size([169, 1024])
(176, 224, 224, 3)
assault 3
torch.Size([176, 1024])
(168, 224, 224, 3)
assault 3
torch.Size([168, 1024])
(166, 224, 224, 3)
assault 3
torch.Size([166, 1024])
(170, 224, 224, 3)
assault 3
torch.Size([170, 1024])
(164, 224, 224, 3)
assault 3
torch.Size([164, 1024])
(159, 224, 224, 3)
assault 3
torch.Size([159, 1024])
(166, 224, 224, 3)
vandalism 11
torch.Size([166, 1024])
(188, 224, 224, 3)
vandalism 11
torch.Size([188, 1024])
(140, 224, 224, 3)
vandalism 11
torch.Size([140, 1024])
(189, 224, 224, 3)
vandalism 11
torch.Size([189,

OutOfMemoryError: CUDA out of memory. Tried to allocate 154.00 MiB. GPU 0 has a total capacty of 15.89 GiB of which 124.12 MiB is free. Process 3013 has 15.77 GiB memory in use. Of the allocated memory 15.06 GiB is allocated by PyTorch, and 426.89 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF